In [7]:
# реализуем чето похожее на бласт
from collections import defaultdict

def build_index(D, k):
    """Создание индекса базы данных"""
    index = defaultdict(list)

    for i in range(len(D) - k + 1):
        kmer = D[i:i+k]
        index[kmer].append(i)

    return dict(index)


def find_seeds(Q, index, k):
    """Поиск так называемого семени совпадений"""
    seeds = []

    for i in range(len(Q) - k + 1):
        kmer = Q[i:i+k]

        if kmer in index:
            for j in index[kmer]:
                seeds.append((i, j, kmer))

    return seeds


def extend(Q, D, qi, di, k, direction, match, mismatch, X):
    """
    Расширение семени влево или вправо
    директион +1 - вправо и наоборот
    """

    seed_score = k * match

    Scur = seed_score
    Smax = seed_score

    history = []

    step = 1

    while True:

        if direction == -1:
            q = qi - step
            d = di - step
        else:
            q = qi + k - 1 + step
            d = di + k - 1 + step

        if q < 0 or d < 0 or q >= len(Q) or d >= len(D):
            break

        if Q[q] == D[d]:
            score = match
        else:
            score = mismatch

        Scur += score

        history.append({
            "q_pos": q,
            "d_pos": d,
            "q_char": Q[q],
            "d_char": D[d],
            "Scur": Scur
        })

        if Scur > Smax:
            Smax = Scur

        if Smax - Scur >= X:
            break

        step += 1

    return Smax, history


def seed_and_extend(Q, D, k=4, match=1, mismatch=-1, gap=-1, X=2):

    index = build_index(D, k)

    seeds = find_seeds(Q, index, k)

    results = []

    for qi, di, kmer in seeds:

        left = extend(Q, D, qi, di, k, -1, match, mismatch, X)
        right = extend(Q, D, qi, di, k, 1, match, mismatch, X)

        results.append({
            "seed": kmer,
            "q_pos": qi,
            "d_pos": di,
            "left": left,
            "right": right
        })

    return index, seeds, results

In [11]:
D = "CTAGGATCCAGGCATACGA"
Q = "GGATCCATTCATTA"

k = 4
X = 2

index, seeds, results = seed_and_extend(Q, D, k=k, X=X)

print("DATABASE INDEX")
print(index)
print()

print("SEEDS FOUND")
for s in seeds:
    print(s)
print()


DATABASE INDEX
{'CTAG': [0], 'TAGG': [1], 'AGGA': [2], 'GGAT': [3], 'GATC': [4], 'ATCC': [5], 'TCCA': [6], 'CCAG': [7], 'CAGG': [8], 'AGGC': [9], 'GGCA': [10], 'GCAT': [11], 'CATA': [12], 'ATAC': [13], 'TACG': [14], 'ACGA': [15]}

SEEDS FOUND
(0, 3, 'GGAT')
(1, 4, 'GATC')
(2, 5, 'ATCC')
(3, 6, 'TCCA')

